# 04 — Account Segmentation

Four-class account segmentation:
1. **Eligible - No Risk**: Growing, low utilization, clean history
2. **Eligible - With Risk**: Active but stress signals present
3. **No Increase Needed**: Stable, no growth demand
4. **Non-Performing**: Delinquent, fraud-flagged, or inactive

Approach: Rule-based labels → Random Forest classifier

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_pipeline
from src import features as feat
from src.models import segmentation as seg

pd.set_option('display.max_columns', 50)

In [ ]:
con, counts = load_pipeline(verbose=True)

## 1. Build Customer Base & Apply Rule-Based Labels

In [ ]:
customer_base = feat.build_customer_base(con)
print(f"customer_base: {customer_base.shape}")

In [ ]:
# Apply rule-based segmentation
customer_segmented = seg.assign_rule_based_segments(customer_base)

print("\n=== Segment Distribution ===")
dist = customer_segmented['segment_name'].value_counts()
print(dist)
print(f"\nTotal: {len(customer_segmented):,}")

In [ ]:
# Segment distribution bar chart (with credit exposure)
seg_summary = customer_segmented.groupby('segment_name').agg(
    account_count=('current_account_nbr', 'count'),
    total_credit_exposure=('cu_crd_line', lambda x: pd.to_numeric(x, errors='coerce').sum()),
    avg_utilization=('ca_current_utilz', lambda x: pd.to_numeric(x, errors='coerce').mean()),
    avg_balance=('cu_cur_balance', lambda x: pd.to_numeric(x, errors='coerce').mean()),
).reset_index()

fig = make_subplots(rows=1, cols=2, subplot_titles=[
    'Account Count by Segment', 'Total Credit Exposure by Segment'
])

colors = [seg.SEGMENT_COLORS.get(s, '#999') for s in seg_summary['segment_name']]

fig.add_trace(go.Bar(
    x=seg_summary['segment_name'], y=seg_summary['account_count'],
    marker_color=colors, name='Accounts'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=seg_summary['segment_name'], y=seg_summary['total_credit_exposure'],
    marker_color=colors, name='Credit Exposure'
), row=1, col=2)

fig.update_layout(title_text='Segment Distribution with Total Credit Exposure',
                  showlegend=False, height=500, width=1000, template='plotly_white')
fig.update_yaxes(title_text='Count', row=1, col=1)
fig.update_yaxes(title_text='Total Credit Line ($)', row=1, col=2)
fig.write_image("../outputs/figures/segment_distribution.png", scale=2)
fig.show()

print("\nSegment Summary:")
print(seg_summary.to_string(index=False))

## 2. Optional: K-Means Clustering Feature

In [ ]:
# Add K-Means cluster as additional feature
customer_clustered, kmeans_model, cluster_scaler = seg.add_kmeans_cluster_feature(
    customer_segmented, n_clusters=6
)
print(f"\nK-Means Cluster Distribution:")
print(customer_clustered['cluster_id'].value_counts().sort_index())

## 3. Train Random Forest Classifier

In [ ]:
# Add cluster_id to classification features
clf_features = seg.CLASSIFICATION_FEATURES + ['cluster_id']

model, report, cm, test_df, used_features = seg.train_segmentation_classifier(
    customer_clustered,
    feature_cols=clf_features,
    test_size=0.25,
    save_path='../outputs/saved_models/rf_segmentation.joblib'
)

## 4. Confusion Matrix

In [ ]:
segment_names = [seg.SEGMENTS[i] for i in sorted(seg.SEGMENTS.keys())]

fig = px.imshow(
    cm,
    labels=dict(x="Predicted", y="Actual", color="Count"),
    x=segment_names, y=segment_names,
    color_continuous_scale='Blues',
    title='Segmentation — Confusion Matrix',
    text_auto=True,
    width=700, height=600
)
fig.update_layout(template='plotly_white')
fig.write_image("../outputs/figures/segmentation_confusion_matrix.png", scale=2)
fig.show()

## 5. Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    'feature': used_features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=True)

fig = px.bar(importance_df.tail(15), x='importance', y='feature', orientation='h',
             title='Random Forest Segmentation — Top 15 Feature Importances',
             color='importance', color_continuous_scale='Greens')
fig.update_layout(height=500, width=700, template='plotly_white', yaxis_title='')
fig.write_image("../outputs/figures/segmentation_feature_importance.png", scale=2)
fig.show()

## 6. Segment Profiles

In [ ]:
# Key metrics by segment
profile_cols = ['cu_bhv_scr', 'cu_crd_bureau_scr', 'ca_current_utilz',
                'avg_monthly_sales_6m', 'cu_nbr_days_dlq', 'ca_nsf_count_lst_12_months',
                'delinquent_cycle_count', 'cu_crd_line']

for col in profile_cols:
    if col in customer_segmented.columns:
        customer_segmented[col] = pd.to_numeric(customer_segmented[col], errors='coerce')

segment_profile = customer_segmented.groupby('segment_name')[profile_cols].mean().round(2)
print("\n=== Average Feature Values by Segment ===")
print(segment_profile.to_string())

## 7. Save Segmented Data

In [ ]:
customer_segmented.to_parquet('../outputs/predictions/customer_segmented.parquet', index=False)
print(f"Saved {len(customer_segmented)} segmented accounts")